# Rare-word sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — the frozen 500-word list | the head of the list is vocabulary, not tokenizer debris |
| Step 2 | `leakage` — near-duplicate audit, quarantine | few enough flags that quarantining leaves the pool intact |
| Step 3 | `sparse_select` — selection dry run | full dose ≥ 30% of val queries, else report and stop |

The method: count how many training sentences carry each source word, freeze the 500
rarest, and for each val query take up to the four rarest listed words it carries. Each
selected word contributes one exemplar — the training example containing it that is
nearest the query — and ordinary cosine kNN fills the rest of the eight prompt slots.

The df floor is the one parameter that decides whether the channel fires. The 500 rarest
words in the whole vocabulary all sit at df 1 and no val query carries them; the floor
raises the list to words rare enough to be marked and common enough to recur.

In [1]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB


In [2]:
%cd /home/prnamhr/projects/Style-Aware-MT
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

/home/prnamhr/projects/Style-Aware-MT

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [3]:
!python3 manage.py build_index --config configs/base_qwen.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 340/340 [01:08<00:00,  4.98it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs


---
## Step 1 — the rarity list

In [4]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

Counting term frequencies over 10860 training sources (zwnj=keep) ...
  22789 pool terms, 718 at df >= 30 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [30, 79]
  df histogram (frozen): {'1': 0, '2': 0, '3': 0, '4-9': 0, '10-99': 500, '100+': 0}
  ZWNJ variant collisions: 25 [('آسود\u200cگی', 'آسودگی'), ('الذین\u200cهم', 'الذینهم'), ('انشآء\u200cالله', 'انشآءالله')]
Wrote results/rarity_train.json and results/rarity_train_top50.tsv


In [5]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
cfg = rarity['config']
print(f"{rarity['n_terms']} pool terms, {rarity['n_eligible']} at df >= {cfg['min_df']} "
      f"-> {rarity['n_frozen']} frozen (requested {cfg['freeze_n']})")
print(f"realized df {rarity['df_observed']}, {rarity['selected_frac']:.1%} of the vocabulary")
print('pool df histogram  :', rarity['df_histogram']['pool'])
print('frozen df histogram:', rarity['df_histogram']['frozen'])

# The list is the 500 rarest above the floor, so it stops wherever the 500th term sits:
# a ceiling on df is an outcome of the freeze, not a parameter.
print(f"n_eligible / freeze_n = {rarity['n_eligible'] / cfg['freeze_n']:.2f}; "
      f"at 1.0 the floor alone decides the list and the ranking selects nothing")

22789 pool terms, 718 at df >= 30 -> 500 frozen (requested 500)
realized df [30, 79], 2.2% of the vocabulary
pool df histogram  : {'1': 11668, '2': 3669, '3': 1809, '4-9': 3357, '10-99': 2113, '100+': 173}
frozen df histogram: {'1': 0, '2': 0, '3': 0, '4-9': 0, '10-99': 500, '100+': 0}
n_eligible / freeze_n = 1.44; at 1.0 the floor alone decides the list and the ranking selects nothing


In [6]:
# Terms tie in df in bulk, and ties fall to total frequency and then to token order.
# If the list concentrates on a few first letters, the token order is choosing it.
initials = pd.Series([t[0] for t, *_ in rarity['terms']]).value_counts()
print(f'{len(initials)} distinct first letters over {rarity["n_frozen"]} terms; '
      f'the top 5 hold {initials.head(5).sum() / rarity["n_frozen"]:.1%}')
initials.head(12).to_frame('terms').T

32 distinct first letters over 500 terms; the top 5 hold 54.2%


,ا,م,ع,ب,ت,س,ل,ن,ک,ف,ج,و
terms,141,44,34,30,22,19,18,18,17,17,16,13


In [7]:
# The check that matters: these should read as marked vocabulary. If the head is broken
# segmentation instead, the df count is ranking tokenizer debris — stop and fix it.
top = pd.read_csv('results/rarity_train_top50.tsv', sep='\t')
top['example'] = top['example'].str.slice(0, 60)
top

,rank,term,df,tf,example
0,0,اظهرت,30,30,و هذا یوم فیه حدّثت الأرض اخبارها و اظهرت کنوزها
1,1,اعدائک,30,30,کأنّ فی قلوبهم اضآء سراج حبّک و مصباح ودّک. لا...
2,2,الندآء,30,30,ثمّ استقمنا علی حبّک علی شأن لا نتوجّه الی دون...
3,3,الوهاب,30,30,إنه لهو المعطی الوهاب
4,4,اینکه,30,30,و شیشه‌ئی خالی نزدش دیده شد مشعر بر اینکه سم خ...
5,5,بعض,30,30,تفضّل حضرة عبدالبهاء وأبان أنّ الأنغام والموسی...
6,6,عندهم,30,30,و باسمک السّخّار بأن تسخّر اهل مملکتک لیقبلنّ ...
7,7,مابین,30,30,نفحات وحی از دونش ممتاز و بیان الهی مابین کتب ...
8,8,مراتب,30,30,اگر ارادهٔ الهی مدد فرماید شرح مبسوطى در آنچه ...
9,9,مقبلا,30,30,لک الحمد بما دللتنی الیک و هدیتنی الی افقک و ا...


### Normalization check — ZWNJ

In [8]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

25 ZWNJ variant collisions


,split spelling,joined spelling
0,آسود‌گی,آسودگی
1,الذین‌هم,الذینهم
2,انشآء‌الله,انشآءالله
3,این‌قدر,اینقدر
4,بیچار‌گان,بیچارگان
5,بی‌خبر,بیخبر
6,بی‌مثال,بیمثال
7,جان‌فزا,جانفزا
8,خون‌ریزی,خونریزی
9,راست‌گو,راستگو


---
## Step 2 — leakage audit

In [9]:
!python manage.py leakage \
    --config configs/sparse_retrieval.yaml \
    --split val \
    --write-quarantine

Auditing 1323 val rows against 10860 pool rows ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 2448.89it/s]
  17/1323 eval rows flagged, 22 pool rows implicated -> results/leakage_val.json
Quarantined 22 pool rows (0.20%) -> data/splits/pool_quarantine.json


In [10]:
leak = json.load(open('results/leakage_val.json'))
print(f"val: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
      f"{leak['n_pool_rows_flagged']} pool rows implicated")
print('  max-cos histogram:', leak['max_cos_histogram'])

pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in leak['flags'][:10]
])

val: 17/1323 eval rows flagged, 22 pool rows implicated
  max-cos histogram: {'0.00-0.50': 0, '0.50-0.70': 0, '0.70-0.80': 0, '0.80-0.85': 0, '0.85-0.90': 506, '0.90-0.95': 816, '0.95-0.97': 1, '0.97-0.99': 0, '0.99-1.01': 0}


,cos,jac_src,jac_tgt,eval,pool
0,0.9508,0.7195,0.2353,امید هست در ظلّ سدرهٴ عنایت الهی تربیت شوید و ...,امید هست در ظلّ سدرهٔ عنایت الهیّه تربیت شوید ...
1,0.9451,0.8319,0.8779,چون که هر روز را امری و هر حین را حکمی مقتضی ل...,چون که هر روز را امری و هر حین را حکمتی مقتضی ...
2,0.9394,0.7059,0.6381,تمسّکوا بحبل الأسباب متوکّلین علی الله مسبّب ا...,تمسّکوا بحبل الأسباب متوکلین على الله مسبّب ال...
3,0.9331,0.7209,0.4286,انّک انت القویّ المقتدر العزیز المتین.,انّک انت المقتدر المتعالی القویّ العزیز العظیم.
4,0.9301,0.7313,0.8491,یا حزب الله مربّی عالم عدل است چه که دارای دو ...,مربّی عالم عدلست چه که دارای دو رکن است مجازات...
5,0.9298,0.5897,0.7500,ابغض النّاس عند الله من یقعد و یطلب.,أبغض الناس عند الله من یقعد ویطلب.
6,0.9267,0.9459,0.4831,انّک انت المقتدر العزیز المهیمن القیّوم.,و انّک انت المقتدر المهیمن العزیز القیّوم.
7,0.9251,0.8611,0.3600,و لا یسأل عمّا یفعل و کلّ عن کلّ یسألون.,انّه لا یسأل عمّا یفعل و کلّ عن کلّ یسألون.
8,0.9209,0.7308,0.7674,انّ ربّک لهو العلیم الحکیم.,انّ ربّک هو العلیم الحکیم.
9,0.9205,0.9048,0.5909,و الحمد لله ربّ العالمین.,الحمد لله ربّ العالمین!


In [11]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

Quarantine data/splits/pool_quarantine.json: dropped 22 of 10860 rows
Embedding 10838 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 339/339 [01:17<00:00,  4.39it/s]
Wrote index to data/knn_index_clean/ : embeddings (10838, 1024), 10838 pairs


---
## Step 3 — the sparse channel

The go/no-go. A query routes to the rare channel as soon as it carries one listed word,
so the numbers below are properties of the list: how many listed words a val query
carries, how often it carries any, and how often it carries four.

In [12]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

Selecting k=8 (up to m=4 rare) for 1323 val sources ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 2451.10it/s]
Dense-only baseline for the redundancy comparison ...
  routes: {'full': 0.1308, 'partial': 0.5896, 'dense': 0.2797}
  rare slots filled: mean 1.514, {'0': 370, '1': 391, '2': 247, '3': 142, '4': 173}
  queries with >= n rare terms: {'1': 0.7203, '2': 0.4248, '3': 0.2381, '4': 0.1308, '5': 0.0733, '6': 0.0355}
  targeted terms served: 1.0
  intra-set cosine: {'sparse': 0.898, 'dense_baseline': 0.9002}
Wrote results/sparse_selection_val.json


In [13]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('rare words/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('targeted served   :', sel['terms_served'])
print('rare slots filled :', sel['n_sparse']['histogram'])
print('intra-set cosine  :', sel['intra_set_similarity'])

routes            : {'full': 0.1308, 'partial': 0.5896, 'dense': 0.2797}
rare words/query — mean 1.653 deciles [0, 0, 0, 1, 1, 1, 2, 2, 3, 4, 10]
share at or above : {'1': 0.7203, '2': 0.4248, '3': 0.2381, '4': 0.1308, '5': 0.0733, '6': 0.0355}
targeted served   : 1.0
rare slots filled : {'0': 370, '1': 391, '2': 247, '3': 142, '4': 173}
intra-set cosine  : {'sparse': 0.898, 'dense_baseline': 0.9002}


In [14]:
# One exemplar per selected word, so a query carrying four listed words fills four slots:
# the full-dose rate should track the share carrying four, unlike under greedy coverage.
EXPECTED = {'mean matches/query': 3.0, '>=1 match': 0.88, 'full dose (4 slots)': 0.34}
FULL_DOSE_GATE = 0.30

now = {
    'mean matches/query': sel['query_terms']['mean'],
    '>=1 match': sel['query_terms']['share_at_or_above']['1'],
    'full dose (4 slots)': sel['route_fractions']['full'],
}
print(pd.DataFrame({'now': now, 'expected': EXPECTED}).to_string())

gap = sel['query_terms']['share_at_or_above']['4'] - now['full dose (4 slots)']
print(f"\nqueries carrying 4+ listed words that did not fill 4 slots: {gap:.1%} "
      f"(every pool example carrying the word was already taken)")

full = now['full dose (4 slots)']
verdict = 'PROCEED' if full >= FULL_DOSE_GATE else 'HOLD'
print(f'full dose {full:.1%} against a {FULL_DOSE_GATE:.0%} gate -> {verdict}')
if verdict == 'HOLD':
    print('Report these numbers; do not generate on this list.')

                        now  expected
mean matches/query   1.6530      3.00
>=1 match            0.7203      0.88
full dose (4 slots)  0.1308      0.34

queries carrying 4+ listed words that did not fill 4 slots: 0.0% (every pool example carrying the word was already taken)
full dose 13.1% against a 30% gate -> HOLD
Report these numbers; do not generate on this list.


In [15]:
import numpy as np
import yaml

from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import SparseRetriever

# Rare picks ran 15.4% shorter than the baseline's exemplars on the old list. A rare arm
# whose prompts are systematically shorter confounds quality with prompt length.
CFG = yaml.safe_load(open('configs/sparse_retrieval.yaml'))
RETR, SPA, RAR = CFG['retrieval'], CFG['sparse'], CFG['rarity']
SRC = [json.loads(ln)['input'] for ln in open('data/splits/val.jsonl') if ln.strip()]

index = RetrievalIndex('data/knn_index_clean', embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(RAR['out']), index, zwnj=RAR['zwnj'], m=SPA['m'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
BASE_SEL = index.retrieve(SRC, k=RETR['k'])

RAR_LEN, COS_LEN = [], []
for t in TRACES:
    rows = set(t['sparse_rows'])
    for r in t['final_rows']:
        (RAR_LEN if r in rows else COS_LEN).append(len(index.pairs[r]['input']))

BASE_CHARS = sum(len(e['input']) for row in BASE_SEL for e in row) / len(TRACES)
ARM_CHARS = (sum(RAR_LEN) + sum(COS_LEN)) / len(TRACES)
for name, v in (('query', [len(x) for x in SRC]), ('rare channel', RAR_LEN),
                ('cosine fill', COS_LEN)):
    a = np.array(v)
    print(f'{name:14s} n={len(a):6d}  mean {a.mean():6.1f}  median {np.median(a):6.1f} chars')
print(f'exemplar chars per prompt: dense {BASE_CHARS:.0f} -> sparse {ARM_CHARS:.0f} '
      f'({ARM_CHARS / BASE_CHARS - 1:+.1%})')

/home/prnamhr/projects/Style-Aware-MT/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading intfloat/multilingual-e5-large-instruct on device: cuda


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2832.16it/s]


query          n=  1323  mean   78.2  median   63.0 chars
rare channel   n=  2003  mean  195.9  median  173.0 chars
cosine fill    n=  8581  mean  198.4  median  180.0 chars
exemplar chars per prompt: dense 1606 -> sparse 1583 (-1.4%)


In [16]:
for min_df in (1, 10, 30, 60):
    lst = f'results/rarity_train_mindf{min_df}.json'
    print(f'--- min_df={min_df}')
    !python3 manage.py rarity --config configs/sparse_retrieval.yaml --min_df {min_df} --out {lst} 2>&1 | grep -E 'frozen|realized'
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --rarity {lst} --out results/sparse_sweep_val_mindf{min_df}.json 2>&1 | grep -E 'routes|rare slots'

--- min_df=1
  22789 pool terms, 22789 at df >= 1 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [1, 1]
  df histogram (frozen): {'1': 500, '2': 0, '3': 0, '4-9': 0, '10-99': 0, '100+': 0}
  routes: {'full': 0.0, 'partial': 0.0144, 'dense': 0.9856}
  rare slots filled: mean 0.014, {'0': 1304, '1': 19, '2': 0, '3': 0, '4': 0}
--- min_df=10
  22789 pool terms, 2286 at df >= 10 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [10, 12]
  df histogram (frozen): {'1': 0, '2': 0, '3': 0, '4-9': 0, '10-99': 500, '100+': 0}
  routes: {'full': 0.003, 'partial': 0.2955, 'dense': 0.7014}
  rare slots filled: mean 0.392, {'0': 928, '1': 294, '2': 82, '3': 15, '4': 4}
--- min_df=30
  22789 pool terms, 718 at df >= 30 -> 500 frozen (requested 500)
  2.2% of the vocabulary, realized df [30, 79]
  df histogram (frozen): {'1': 0, '2': 0, '3': 0, '4-9': 0, '10-99': 500, '100+': 0}
  routes: {'full': 0.1308, 'partial': 0.5896, 'dense': 0.2797}
  rare slots filled: m

In [17]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| carries', ex['trace']['query_terms'])
    print('  served', ex['trace']['served_terms'], f"+ {ex['trace']['n_knn']} kNN")
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()

QUERY : جواهر الأسرار فی معارج الأسفار لمن اراد ان یتقرّب بالله المقتدر الغفّار فهنیاً للأبرار الّ
  route partial | carries ['اراد', 'بالله']
  served ['اراد', 'بالله'] + 6 kNN
   - و بهدایت کبری و ربوبیّت عظمی مبعوث شوند که تا قلوب مشتاقین و حقایق صافین را بالهامات غیبیّ
   - فأمطر من سحاب فیض فضلک ما تطهّر به افئدة عبادک عمّا یحجبهم عن النّظر الی وجهک و یمنعهم عن 
   - اللّهمّ انّی اسألک بالحرف الّتی اذا خرجت من فم مشیّتک ماجت البحار و هاجت الأریاح و ظهرت ال
   - من شرب من الکأس الّتی تدور بها ید رحمتک ینقطع عن دونک و ینجذب بکلمة منه عبادک الّذین رقدوا
   - لو تعرف ما نزّل من قلمی و تطّلع علی خزائن امری و لآلئ اسراری فی بحور اسمائی و اواعی کلماتی
   - طوبی لک بما کنت سائراً فی بلاد الله و کنت آیة الفرح و الاطمینان لأهل البهآء الّذین انقطعوا
   - تفکّروا فی هذه الآیة ثم انصفوا بالله لعلّ تجدون لئالئ الأسرار من البحر الذی تموّج باسمی ال
   - لو تسمع صریر القلم الأعلی و هدیر ورقآء البقآء علی افنان سدرة المنتهی فی ذکر الله موجد الأس

QUERY : هو العلیّ الأعلی.
  route partial | carries [